In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:
%%RecordEvent
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
DATA_ROOT = "/home/colinc/code/tpch/data/tpch_15"#"/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}
import pandas as pd

In [4]:

### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)



Index(['L_ORDERKEY', 'L_PARTKEY', 'L_SUPPKEY', 'L_LINENUMBER', 'L_QUANTITY',
       'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX', 'L_RETURNFLAG',
       'L_LINESTATUS', 'L_SHIPDATE', 'L_COMMITDATE', 'L_RECEIPTDATE',
       'L_SHIPINSTRUCT', 'L_SHIPMODE', 'L_COMMENT', 'L_DUMMY'],
      dtype='object')


In [5]:

### cell 1 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [6]:
%%time
### cell 2 ###

### cell 2 (optimized for cudf) ###

# Filter lineitem and extract unique order keys
fline_keys = lineitem.loc[lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE, 'L_ORDERKEY']

# Filter orders by date range
orders_filtered = orders.loc[
    (orders.O_ORDERDATE >= '1993-08-01') &
    (orders.O_ORDERDATE < '1993-11-01')
]

# Select orders matching the filtered lineitem keys
matched_orders = orders_filtered.loc[
    orders_filtered.O_ORDERKEY.isin(fline_keys)
]

# Count distinct orders by priority
total = matched_orders.groupby('O_ORDERPRIORITY', as_index=False)['O_ORDERKEY'].count()

CPU times: user 464 ms, sys: 208 ms, total: 672 ms
Wall time: 947 ms


Generating '/var/tmp/nsys-report-7696.qdstrm'
[1/2] [========================100%] o4_mini_high_q04_2025-07-11T19:54:08.nsys-rep
[2/2] [========================100%] o4_mini_high_q04_2025-07-11T19:54:08.sqlite
Generated:
	/home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_q04_2025-07-11T19:54:08.nsys-rep
	/home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_q04_2025-07-11T19:54:08.sqlite
Generating SQLite file /home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_q04_2025-07-11T19:54:08.sqlite from /home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_q04_2025-07-11T19:54:08.nsys-rep
Processing [/home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_q04_2025-07-11T19:54:08.sqlite] with [/opt/nvidia/nsight-systems/2025.3.1/host-linux-x64/reports/nvtx_sum.py]... 

 ** NVTX Range Summary (nvtx_sum):

 Time (%)  Total Time (ns)  Instances   Avg (ns)     Med (ns)    Min (ns)   Max (ns)   StdDev (ns)   Style                           Range

In [7]:
%%time
#original
date1 = pd.Timestamp("1993-11-01")
date2 = pd.Timestamp("1993-08-01")
lsel = lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE
osel = (orders.O_ORDERDATE < date1) & (orders.O_ORDERDATE >= date2)
flineitem = lineitem[lsel]
forders = orders[osel]
jn = forders[forders["O_ORDERKEY"].isin(flineitem["L_ORDERKEY"])]
total = (
    jn.groupby("O_ORDERPRIORITY", as_index=False)["O_ORDERKEY"].count()
    # skip sort when Mars enables sort in groupby
    # .sort_values(["O_ORDERPRIORITY"])
)

CPU times: user 7.89 s, sys: 4.27 s, total: 12.2 s
Wall time: 9.4 s


Generating '/var/tmp/nsys-report-0df7.qdstrm'
[1/2] [========================100%] o4_mini_high_q04_2025-07-11T19:41:45.nsys-rep
[2/2] [========================100%] o4_mini_high_q04_2025-07-11T19:41:45.sqlite
Generated:
	/home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_q04_2025-07-11T19:41:45.nsys-rep
	/home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_q04_2025-07-11T19:41:45.sqlite
Generating SQLite file /home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_q04_2025-07-11T19:41:45.sqlite from /home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_q04_2025-07-11T19:41:45.nsys-rep
Processing [/home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_q04_2025-07-11T19:41:45.sqlite] with [/opt/nvidia/nsight-systems/2025.3.1/host-linux-x64/reports/nvtx_sum.py]... 

 ** NVTX Range Summary (nvtx_sum):

 Time (%)  Total Time (ns)  Instances    Avg (ns)     Med (ns)    Min (ns)    Max (ns)   StdDev (ns)    Style                           Ra

In [8]:
### cell 3 ###

total.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   O_ORDERPRIORITY  5 non-null      object
 1   O_ORDERKEY       5 non-null      int64
dtypes: int64(1), object(1)
memory usage: 106.0+ bytes
